In [ ]:
!pip install -q transformers torch

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Traditional ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Deep Learning
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Flatten,
    Dense,
    SimpleRNN,
    LSTM,
    Dropout
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import EarlyStopping

# Hugging Face
from transformers import pipeline

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [ ]:
texts = [

    # -------------------------
    # POSITIVE
    # -------------------------

    "I love this product",
    "This product is excellent",
    "I really enjoyed this movie",
    "The movie was fantastic",
    "I am very happy with the service",
    "The service was excellent",
    "I love this course",
    "The course was very useful",
    "I enjoyed learning this topic",
    "This is an amazing experience",

    "The food was delicious",
    "I really liked the food",
    "The restaurant was wonderful",
    "The staff was very friendly",
    "I had a great experience",
    "This software is very useful",
    "The application works perfectly",
    "I love using this application",
    "The training was excellent",
    "I learned a lot from this training",

    "The teacher explained everything clearly",
    "The lecture was interesting",
    "The project was exciting",
    "I am satisfied with the result",
    "The results are excellent",
    "This is a great idea",
    "I strongly recommend this product",
    "The customer service was helpful",
    "The experience was enjoyable",
    "I am extremely happy",

    # -------------------------
    # NEGATIVE
    # -------------------------

    "I hate this product",
    "This product is terrible",
    "I really disliked this movie",
    "The movie was boring",
    "I am unhappy with the service",
    "The service was terrible",
    "I hate this course",
    "The course was useless",
    "I disliked learning this topic",
    "This was a horrible experience",

    "The food was terrible",
    "I really disliked the food",
    "The restaurant was disappointing",
    "The staff was rude",
    "I had a bad experience",
    "This software is useless",
    "The application does not work",
    "I hate using this application",
    "The training was disappointing",
    "I learned nothing from this training",

    "The teacher explained everything poorly",
    "The lecture was boring",
    "The project was frustrating",
    "I am disappointed with the result",
    "The results are terrible",
    "This is a bad idea",
    "I do not recommend this product",
    "The customer service was unhelpful",
    "The experience was unpleasant",
    "I am extremely unhappy"
]

# 1 = Positive
# 0 = Negative

labels = [1] * 30 + [0] * 30

df = pd.DataFrame({
    "text": texts,
    "sentiment": labels
})

print(df.head())
print()
print(df["sentiment"].value_counts())

                               text  sentiment
0               I love this product          1
1         This product is excellent          1
2       I really enjoyed this movie          1
3           The movie was fantastic          1
4  I am very happy with the service          1

sentiment
1    30
0    30
Name: count, dtype: int64


In [ ]:
print("Total sentences:", len(df))

print("\nPositive:", sum(df["sentiment"] == 1))
print("Negative:", sum(df["sentiment"] == 0))

Total sentences: 60

Positive: 30
Negative: 30


In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment"],
    test_size=0.20,
    random_state=42,
    stratify=df["sentiment"]
)

print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

Training samples: 48
Testing samples: 12


In [ ]:
#traditional ml using tf-idf

Text
 ↓
TF-IDF
 ↓
Numerical Features
 ↓
Logistic Regression
 ↓
Positive / Negative

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

print("Training matrix:", X_train_tfidf.shape)

Training matrix: (48, 178)


In [ ]:
ml_model = LogisticRegression(
    max_iter=1000
)

ml_model.fit(
    X_train_tfidf,
    y_train
)

LogisticRegression(max_iter=1000)

In [ ]:
ml_pred = ml_model.predict(
    X_test_tfidf
)

ml_probability = ml_model.predict_proba(
    X_test_tfidf
)[:, 1]

ml_accuracy = accuracy_score(
    y_test,
    ml_pred
)

print(
    "Traditional ML Accuracy:",
    ml_accuracy
)

Traditional ML Accuracy: 0.25


In [ ]:
#ANN

Text
 ↓
Tokenization
 ↓
Padding
 ↓
Embedding
 ↓
Flatten
 ↓
Dense(16, ReLU)
 ↓
Dense(8, ReLU)
 ↓
Dense(1, Sigmoid)
 ↓
Sentiment

In [ ]:
tokenizer = Tokenizer(
    num_words=1000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(
    X_train_text
)

vocab_size = len(
    tokenizer.word_index
) + 1

print("Vocabulary size:", vocab_size)

Vocabulary size: 76


In [ ]:
X_train_seq = tokenizer.texts_to_sequences(
    X_train_text
)

X_test_seq = tokenizer.texts_to_sequences(
    X_test_text
)

In [ ]:
max_length = 10

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post"
)

print(X_train_pad.shape)

(48, 10)


In [ ]:
ann_model = Sequential([

    Input(shape=(max_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=16
    ),

    Flatten(),

    Dense(
        16,
        activation="relu"
    ),

    Dense(
        8,
        activation="relu"
    ),

    Dropout(0.2),

    Dense(
        1,
        activation="sigmoid"
    )
])

ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 16)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 160)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         2,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,937 (15.38 KB)

 Trainable params: 3,937 (15.38 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
ann_history = ann_model.fit(

    X_train_pad,
    y_train,

    epochs=30,

    batch_size=8,

    validation_split=0.2,

    verbose=1
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 134ms/step - accuracy: 0.5263 - loss: 0.6890 - val_accuracy: 0.4000 - val_loss: 0.7063
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5526 - loss: 0.6854 - val_accuracy: 0.4000 - val_loss: 0.7086
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5789 - loss: 0.6839 - val_accuracy: 0.4000 - val_loss: 0.7117
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.5789 - loss: 0.6755 - val_accuracy: 0.4000 - val_loss: 0.7102
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.6316 - loss: 0.6744 - val_accuracy: 0.4000 - val_loss: 0.7117
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.6053 - loss: 0.6599 - val_accuracy: 0.4000 - val_loss: 0.7114
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7105 - loss: 0.6534 - val_accuracy: 0.4000 - val_loss: 0.7125
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.6842 - loss: 0.6543 - val_accuracy: 0.4000 - val_loss: 0.7150

In [ ]:
ann_loss, ann_accuracy = ann_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print(
    "ANN Accuracy:",
    ann_accuracy
)

ANN Accuracy: 0.4166666567325592


Sentence
   ↓
Tokens
   ↓
Embedding
   ↓
SimpleRNN
   ↓
Dense
   ↓
Sigmoid
   ↓
Sentiment

In [ ]:
rnn_model = Sequential([

    Input(shape=(max_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=16
    ),

    SimpleRNN(
        16
    ),

    Dense(
        8,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

rnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 10, 16)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,889 (7.38 KB)

 Trainable params: 1,889 (7.38 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
rnn_history = rnn_model.fit(

    X_train_pad,
    y_train,

    epochs=30,

    batch_size=8,

    validation_split=0.2,

    verbose=1
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.4737 - loss: 0.6960 - val_accuracy: 0.4000 - val_loss: 0.6834
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5263 - loss: 0.6877 - val_accuracy: 0.4000 - val_loss: 0.6853
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.5789 - loss: 0.6784 - val_accuracy: 0.4000 - val_loss: 0.6859
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.6316 - loss: 0.6674 - val_accuracy: 0.5000 - val_loss: 0.6875
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6579 - loss: 0.6573 - val_accuracy: 0.5000 - val_loss: 0.6903
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.6842 - loss: 0.6407 - val_accuracy: 0.5000 - val_loss: 0.6908
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.6842 - loss: 0.6226 - val_accuracy: 0.5000 - val_loss: 0.6922
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.7105 - loss: 0.5988 - val_accuracy: 0.5000 - val_loss: 0.6952

In [ ]:
rnn_loss, rnn_accuracy = rnn_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print(
    "RNN Accuracy:",
    rnn_accuracy
)

RNN Accuracy: 0.3333333432674408


Sentence
   ↓
Tokenization
   ↓
Embedding
   ↓
LSTM
   ↓
Dense
   ↓
Sigmoid
   ↓
Sentiment

In [ ]:
lstm_model = Sequential([

    Input(shape=(max_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=16
    ),

    LSTM(
        16
    ),

    Dense(
        8,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 10, 16)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,473 (13.57 KB)

 Trainable params: 3,473 (13.57 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
lstm_history = lstm_model.fit(

    X_train_pad,
    y_train,

    epochs=30,

    batch_size=8,

    validation_split=0.2,

    verbose=1
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 291ms/step - accuracy: 0.5263 - loss: 0.6931 - val_accuracy: 0.4000 - val_loss: 0.6974
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.5263 - loss: 0.6925 - val_accuracy: 0.4000 - val_loss: 0.6986
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5263 - loss: 0.6920 - val_accuracy: 0.4000 - val_loss: 0.6984
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5263 - loss: 0.6915 - val_accuracy: 0.4000 - val_loss: 0.6988
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5263 - loss: 0.6913 - val_accuracy: 0.4000 - val_loss: 0.6990
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.5263 - loss: 0.6909 - val_accuracy: 0.4000 - val_loss: 0.7001
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5263 - loss: 0.6901 - val_accuracy: 0.4000 - val_loss: 0.7007
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5263 - loss: 0.6893 - val_accuracy: 0.4000 - val_loss: 0.7015

In [ ]:
lstm_loss, lstm_accuracy = lstm_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print(
    "LSTM Accuracy:",
    lstm_accuracy
)

LSTM Accuracy: 0.3333333432674408


In [ ]:
transformer_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
sentence = "I really enjoyed this course"

result = transformer_model(
    sentence
)

print(result)

[{'label': 'POSITIVE', 'score': 0.9998726844787598}]


In [ ]:
def predict_all_models(sentence):

    print("=" * 60)
    print("INPUT SENTENCE")
    print("=" * 60)

    print(sentence)

    print("\n")


    # ---------------------------------
    # Traditional ML
    # ---------------------------------

    sentence_tfidf = tfidf.transform(
        [sentence]
    )

    ml_prob = ml_model.predict_proba(
        sentence_tfidf
    )[0][1]

    ml_label = (
        "Positive"
        if ml_prob >= 0.5
        else "Negative"
    )


    # ---------------------------------
    # Neural network preprocessing
    # ---------------------------------

    sentence_seq = tokenizer.texts_to_sequences(
        [sentence]
    )

    sentence_pad = pad_sequences(
        sentence_seq,
        maxlen=max_length,
        padding="post"
    )


    # ---------------------------------
    # ANN
    # ---------------------------------

    ann_prob = ann_model.predict(
        sentence_pad,
        verbose=0
    )[0][0]

    ann_label = (
        "Positive"
        if ann_prob >= 0.5
        else "Negative"
    )


    # ---------------------------------
    # RNN
    # ---------------------------------

    rnn_prob = rnn_model.predict(
        sentence_pad,
        verbose=0
    )[0][0]

    rnn_label = (
        "Positive"
        if rnn_prob >= 0.5
        else "Negative"
    )


    # ---------------------------------
    # LSTM
    # ---------------------------------

    lstm_prob = lstm_model.predict(
        sentence_pad,
        verbose=0
    )[0][0]

    lstm_label = (
        "Positive"
        if lstm_prob >= 0.5
        else "Negative"
    )


    # ---------------------------------
    # Transformer
    # ---------------------------------

    transformer_result = transformer_model(
        sentence
    )[0]

    transformer_label = (
        "Positive"
        if transformer_result["label"] == "POSITIVE"
        else "Negative"
    )

    transformer_score = transformer_result[
        "score"
    ]


    # ---------------------------------
    # DISPLAY
    # ---------------------------------

    results = pd.DataFrame({

        "Model": [
            "Traditional ML",
            "ANN",
            "RNN",
            "LSTM",
            "Transformer"
        ],

        "Prediction": [
            ml_label,
            ann_label,
            rnn_label,
            lstm_label,
            transformer_label
        ],

        "Score": [
            ml_prob,
            ann_prob,
            rnn_prob,
            lstm_prob,
            transformer_score
        ]
    })

    print(results)

    return results

In [ ]:
sentence = "I really enjoyed this course"

results = predict_all_models(
    sentence
)

INPUT SENTENCE
I really enjoyed this course


            Model Prediction     Score
0  Traditional ML   Positive  0.617517
1             ANN   Positive  0.766313
2             RNN   Positive  0.953553
3            LSTM   Positive  0.986358
4     Transformer   Positive  0.999873


In [ ]:
sentence = "The product was excellent and I am very happy"

predict_all_models(sentence)

INPUT SENTENCE
The product was excellent and I am very happy


            Model Prediction     Score
0  Traditional ML   Positive  0.668892
1             ANN   Positive  0.928299
2             RNN   Positive  0.992437
3            LSTM   Positive  0.983410
4     Transformer   Positive  0.999888


,Model,Prediction,Score
0,Traditional ML,Positive,0.668892
1,ANN,Positive,0.928299
2,RNN,Positive,0.992437
3,LSTM,Positive,0.983410
4,Transformer,Positive,0.999888


In [ ]:
#analysis

In [ ]:
#TF-IDF represents words statistically; it does not truly understand negation.
sentence = "I do not like this product"

tfidf_input = tfidf.transform([sentence])

prediction = ml_model.predict(tfidf_input)[0]

print("Sentence:", sentence)

if prediction == 1:
    print("TF-IDF Prediction: Positive")
else:
    print("TF-IDF Prediction: Negative")

print("Expected: Negative")

Sentence: I do not like this product
TF-IDF Prediction: Negative
Expected: Negative


In [ ]:
sentence = "I thought the movie was good, but the ending was terrible"

seq = tokenizer.texts_to_sequences([sentence])

padded = pad_sequences(
    seq,
    maxlen=max_length,
    padding="post"
)

probability = rnn_model.predict(
    padded,
    verbose=0
)[0][0]

prediction = (
    "Positive"
    if probability >= 0.5
    else "Negative"
)

print("Sentence:", sentence)
print("RNN Prediction:", prediction)
print("Probability:", probability)
print("Expected: Negative")

Sentence: I thought the movie was good, but the ending was terrible
RNN Prediction: Positive
Probability: 0.850135
Expected: Negative


In [ ]:
#RNN has sequence memory, but a small RNN may still struggle to learn complex relationships in a sentence.

In [ ]:
sentence = "I expected the course to be boring, but it was excellent"

seq = tokenizer.texts_to_sequences([sentence])

padded = pad_sequences(
    seq,
    maxlen=max_length,
    padding="post"
)

probability = lstm_model.predict(
    padded,
    verbose=0
)[0][0]

prediction = (
    "Positive"
    if probability >= 0.5
    else "Negative"
)

print("Sentence:", sentence)
print("LSTM Prediction:", prediction)
print("Probability:", probability)
print("Expected: Positive")

Sentence: I expected the course to be boring, but it was excellent
LSTM Prediction: Positive
Probability: 0.9667219
Expected: Positive


The model needs to understand that "boring" describes the expectation, not the final experience.
Even LSTM does not automatically understand this perfectly, especially when trained on only a few dozen examples.
======================
LSTM has better memory mechanisms, but it is still a learned model. If the training data does not contain enough examples of this linguistic pattern, it can fail.

In [ ]:
test_cases = [

    ("I do not like this product", "Negative"),

    ("This movie is not good", "Negative"),

    ("I do not hate this product", "Positive"),

    ("The movie was good but the ending was terrible", "Negative"),

    ("The product looks great but works terribly", "Negative"),

    ("I expected the course to be boring but it was excellent", "Positive"),

    ("The course was difficult but very useful", "Positive"),

    ("Although the product is expensive, I love its quality", "Positive"),

    ("Great, another software crash", "Negative"),

    ("I thought the product would be excellent but I was disappointed", "Negative")
]